# Four-model ink ensemble on the campaign-40 training segments

Segments: every training scroll of `campaign_archs_40.py` arm `holdout_n96_combined_surface_norm_nnpu` (`BASE_SCROLL_DICT`, 24 segments over 12 domains).

1. Setup — choose the third ensemble member with `SLOT3` (`"ink9-42"` = ink_9um seed 42, fast; `"v8in-1447"` = PHerc1447 fine-tune, slow) and the v8-in tile stride with `ve.V8IN_STRIDE`. Seed 43, dense_native and Hecate are always used.
2. Download + assemble the missing zarrs with `assemble_training_segments.py` (no norm / surface supervision recompute; existing masks are backed up first).
3. One cell per segment, then `render_all()` for an unattended run (skips finished segments, logs failures).

Saved (not displayed) to `output/test_visualizations/villa_ensemble/training_c40/`: `<name>_4models_d+0.png` (2x2 grid) and `<name>_ensemble_d+0.png` (ensemble over the composite view with a 1 cm bar), all at full resolution.

In [1]:
import os
import gc
import re
import shutil
import subprocess
import sys
import time

if sys.executable.startswith("/opt/villa-venv"):
    raise RuntimeError("select the system Python 3.10 kernel (/usr/bin/python3); /opt/villa-venv is only used "
                       "internally for model inference")

REPO = "/vesuvius"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import importlib
from utils import villa_ensemble as ve
ve = importlib.reload(ve)
import assemble_training_segments as ats

# third ensemble member: "ink9-42" (ink_9um hybrid_3d2d seed 42) or "v8in-1447" (PHerc1447 fine-tune, much slower)
SLOT3 = "v8in-1447"
ve.set_slot3(SLOT3)
# v8in tile stride: None = its default 21; 32 is 2.3x faster for a small fidelity loss (see utils/villa_ensemble.py)
ve.V8IN_STRIDE = 32

ZARR_DIR = ats.ZARR_DIR
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble", "training_c40")
MASK_BACKUP_DIR = os.path.join(REPO, "backup_masks_training_c40")
ASSEMBLE_LOG = os.path.join(REPO, "_ves_tmp", "assemble_training_c40.log")
DEPTHS = (0,)
WRITE_DEPTH = 0
DIRECTION = "forward"
COMPILE = False
TAG = f"d{WRITE_DEPTH:+d}"
MIN_FREE_GB = 20

# campaign_archs_40.BASE_SCROLL_DICT (training scrolls of holdout_n96_combined_surface_norm_nnpu)
TRAIN_SCROLL_DICT = {
    "pherc0139": [20260115000000, 20260317000000, 20250223000000, 20250108000005, 20260112000000,
                  20260126000000, 20250831000000, 20260108000000, 20260302000000],
    "pherc0172": [20251111010954, 20251112000002],
    "pherc1667": [20240304141531, 20240304144031, 20231201215900],
    "pherc0009b": [20250919125754],
    "phercparis4": [20231210121321],
    "pherc0500p2": [20250628074500],
    "pherc0814": [20260226000000],
    "phercparis2_fr143": [20230301213755, 20230205142449],
    "pherc51cr4_fr8": [20231205222200],
    "phercparis1_fr34": [20230301213423],
    "pherc0343p": [20250511003658],
    "pherc0841": [20260221022814],
}
SEG_NAME = {zid: name for name, _seg, zid in ats.SEGMENTS}
FRAGMENTS = {SEG_NAME[str(zid)]: str(zid) for ids in TRAIN_SCROLL_DICT.values() for zid in ids}
DOMAIN = {str(zid): domain for domain, ids in TRAIN_SCROLL_DICT.items() for zid in ids}
# render pixel size: the 9.362 µm training grid, except level-2 pooled fine scans and the 9.366 µm p841 crop
ve.EXTRA_NATIVE_UM.update({zid: 9.362 for zid in FRAGMENTS.values()})
ve.EXTRA_NATIVE_UM.update({"20240304141531": 9.596, "20240304144031": 9.596, "20231210121321": 9.6,
                           "20260221022814": 9.366})
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm", "ink9-42": "ink_9um seed 42",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-43": "ink_9um seed 43"}


def is_assembled(zid):
    zpath = os.path.join(ZARR_DIR, f"{zid}.zarr")
    return os.path.isfile(os.path.join(zpath, ".zarray")) and os.path.isfile(os.path.join(REPO, "masks", f"{zid}.png"))


def outputs(name):
    return [os.path.join(OUT_DIR, f"{name}_{kind}_{TAG}.png") for kind in ("4models", "ensemble")]


print(f"models: {[short for short, _ in ve.MODELS]} | {len(FRAGMENTS)} segments | zarrs: {ZARR_DIR} | out: {OUT_DIR}")

models: ['dnative', 'hecate', 'v8in-1447', 'ink9-43'] | 24 segments | zarrs: /vesuvius/ves_zarrs2 | out: /vesuvius/output/test_visualizations/villa_ensemble/training_c40


In [2]:
# --- download + assemble the missing training zarrs (resumable: finished segments are skipped) ---
from tqdm.auto import tqdm

todo = [name for name, zid in FRAGMENTS.items() if not is_assembled(zid)]
print(f"{len(FRAGMENTS) - len(todo)} assembled, {len(todo)} to do: {todo}")
print(f"free disk: {shutil.disk_usage(REPO).free / 1e9:.0f} GB")
if todo:
    if shutil.disk_usage(REPO).free / 1e9 < MIN_FREE_GB:
        raise RuntimeError(f"less than {MIN_FREE_GB} GB free")
    # the assembler rewrites masks/<id>.png; keep the current ones
    os.makedirs(MASK_BACKUP_DIR, exist_ok=True)
    for name in todo:
        source = os.path.join(REPO, "masks", f"{FRAGMENTS[name]}.png")
        backup = os.path.join(MASK_BACKUP_DIR, f"{FRAGMENTS[name]}.png")
        if os.path.isfile(source) and not os.path.isfile(backup):
            shutil.copy2(source, backup)
    command = [sys.executable, "-u", "assemble_training_segments.py", "--only", ",".join(todo),
               "--skip-norm", "--skip-surface", "--concurrent-fragments", "3"]
    shown = re.compile(r"\[done\]|\[FAIL\]|\[1/3\]|wrote|Error|WARN|integrity|SUMMARY", re.I)
    bar = tqdm(total=len(todo), desc="assemble", unit="segment")
    with open(ASSEMBLE_LOG, "a", encoding="utf-8") as log, subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("[done]", "[FAIL]")):
                bar.update(1)
            if shown.search(line):
                bar.write(line.rstrip())
        proc.wait()
    bar.close()
    print(f"assembler exit {proc.returncode}; full log: {ASSEMBLE_LOG}")

for name, zid in FRAGMENTS.items():
    ok, msg = ats._zarr_integrity(zid) if is_assembled(zid) else (False, "MISSING")
    print(f"  {'OK  ' if ok else 'FAIL'} {name:<14} {zid}  {msg}")
print(f"free disk: {shutil.disk_usage(REPO).free / 1e9:.0f} GB")

24 assembled, 0 to do: []
free disk: 121 GB
  OK   w044           20260115000000  cov=1.00
  OK   w035           20260317000000  cov=1.00
  OK   w059           20250223000000  cov=1.00
  OK   w030           20250108000005  cov=1.00
  OK   w043           20260112000000  cov=1.00
  OK   w045           20260126000000  cov=1.00
  OK   w040           20250831000000  cov=1.00
  OK   w041           20260108000000  cov=1.00
  OK   w039           20260302000000  cov=1.00
  OK   w068           20251111010954  cov=1.00
  OK   w087           20251112000002  cov=1.00
  OK   w013           20240304141531  cov=1.00
  OK   w018           20240304144031  cov=1.00
  OK   cr1fr3         20231201215900  cov=1.00
  OK   p9b_487        20250919125754  cov=1.00
  OK   paris4         20231210121321  cov=1.00
  OK   500P2_front    20250628074500  cov=1.00
  OK   seg46527       20260226000000  cov=1.00
  OK   paris2_fr143   20230301213755  cov=1.00
  OK   paris2_fr47    20230205142449  cov=1.00
  OK   scroll6_f

In [3]:
def render(name, force=False):
    scroll_id = FRAGMENTS[name]
    grid_path, ens_path = outputs(name)
    if not force and os.path.isfile(grid_path) and os.path.isfile(ens_path):
        print(f"[skip] {name}: already rendered")
        return
    if not is_assembled(scroll_id):
        raise RuntimeError(f"{name} ({scroll_id}) is not assembled; run the assembly cell")
    t0 = time.time()
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, DEPTHS, [DIRECTION], compile_model=COMPILE)
    print("[ensemble] stretching + z-score ensemble ...", flush=True)
    stretched, ensemble, stats = ve.ensemble(scroll_id, DEPTHS, DIRECTION, WRITE_DEPTH)
    label = f"{DOMAIN[scroll_id]} {name}"
    ve.save_grid([(f"{label}  {TITLES[k]}  {TAG} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2, grid_path)
    ve.save_ensemble_figure(scroll_id, ensemble, f"{label}  4-model ensemble (mean z-score)  {TAG} {DIRECTION}",
                            ens_path)
    del stretched, ensemble, stats
    gc.collect()
    print(f"[saved] {grid_path}\n[saved] {ens_path}\n[done] {name} in {time.time() - t0:.0f}s")


def render_all(names=None):
    """unattended: render every segment, logging failures instead of stopping; finished maps/figures are cached"""
    failed = []
    for name in names or FRAGMENTS:
        try:
            render(name)
        except Exception as exc:
            failed.append(name)
            print(f"[FAILED] {name}: {exc}", flush=True)
    print(f"[render_all] {len(failed)} failed: {failed}")

In [4]:
# --- PHerc0814 seg46527 (sanity: known ink) ---
render("seg46527")

[skip] seg46527: already rendered


In [5]:
# --- PHerc0139 w044 ---
render("w044")

20260115000000: 100%|██████████| 1.00/1 maps [21:26<00:00] , v8in-loo-w062__L28s2st32.tif 41260/41260 patches

[villa] 20260115000000: 1 maps in 1287s -> /vesuvius/output/villa_ensemble_maps/20260115000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w044_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w044_ensemble_d+0.png
[done] w044 in 1389s


In [6]:
# --- PHerc0139 w035 ---
render("w035")

20260317000000: 100%|██████████| 3.00/3 maps [17:15<00:00] , hybrid_3d2d-seed43__L28s6.tif 6456/6456 patches

[villa] 20260317000000: 3 maps in 1035s -> /vesuvius/output/villa_ensemble_maps/20260317000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w035_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w035_ensemble_d+0.png
[done] w035 in 1095s


In [7]:
# --- PHerc0139 w059 ---
render("w059")

20250223000000: 100%|██████████| 4.00/4 maps [42:29<00:00] , hybrid_3d2d-seed43__L28s6.tif 15363/15363 patches

[villa] 20250223000000: 4 maps in 2549s -> /vesuvius/output/villa_ensemble_maps/20250223000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w059_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w059_ensemble_d+0.png
[done] w059 in 2706s


In [8]:
# --- PHerc0139 w030 ---
render("w030")

20250108000005: 100%|██████████| 4.00/4 maps [32:05<00:00] , hybrid_3d2d-seed43__L28s6.tif 11412/11412 patches

[villa] 20250108000005: 4 maps in 1926s -> /vesuvius/output/villa_ensemble_maps/20250108000005
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w030_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w030_ensemble_d+0.png
[done] w030 in 2044s


In [9]:
# --- PHerc0139 w043 ---
render("w043")

20260112000000: 100%|██████████| 4.00/4 maps [30:56<00:00] , hybrid_3d2d-seed43__L28s6.tif 11006/11006 patches

[villa] 20260112000000: 4 maps in 1857s -> /vesuvius/output/villa_ensemble_maps/20260112000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w043_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w043_ensemble_d+0.png
[done] w043 in 1971s


In [10]:
# --- PHerc0139 w045 ---
render("w045")

20260126000000: 100%|██████████| 4.00/4 maps [30:49<00:00] , hybrid_3d2d-seed43__L28s6.tif 10950/10950 patches

[villa] 20260126000000: 4 maps in 1850s -> /vesuvius/output/villa_ensemble_maps/20260126000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w045_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w045_ensemble_d+0.png
[done] w045 in 1961s


In [11]:
# --- PHerc0139 w040 ---
render("w040")

20250831000000: 100%|██████████| 4.00/4 maps [31:10<00:00] , hybrid_3d2d-seed43__L28s6.tif 11076/11076 patches

[villa] 20250831000000: 4 maps in 1871s -> /vesuvius/output/villa_ensemble_maps/20250831000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w040_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w040_ensemble_d+0.png
[done] w040 in 1984s


In [12]:
# --- PHerc0139 w041 ---
render("w041")

20260108000000: 100%|██████████| 4.00/4 maps [30:47<00:00] , hybrid_3d2d-seed43__L28s6.tif 10920/10920 patches

[villa] 20260108000000: 4 maps in 1848s -> /vesuvius/output/villa_ensemble_maps/20260108000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w041_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w041_ensemble_d+0.png
[done] w041 in 1959s


In [13]:
# --- PHerc0139 w039 ---
render("w039")

20260302000000: 100%|██████████| 4.00/4 maps [29:37<00:00] , hybrid_3d2d-seed43__L28s6.tif 10514/10514 patches

[villa] 20260302000000: 4 maps in 1777s -> /vesuvius/output/villa_ensemble_maps/20260302000000
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w039_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w039_ensemble_d+0.png
[done] w039 in 1904s


In [14]:
# --- PHerc0172 w068 (7.91 µm, resampled) ---
render("w068")

20251111010954: 100%|██████████| 4.00/4 maps [1:30:54<00:00] , hybrid_3d2d-seed43__L28s6.tif 31964/31964 patches

[villa] 20251111010954: 4 maps in 5455s -> /vesuvius/output/villa_ensemble_maps/20251111010954
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w068_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w068_ensemble_d+0.png
[done] w068 in 6224s


In [15]:
# --- PHerc0172 w087 (7.91 µm, resampled) ---
render("w087")

20251112000002: 100%|██████████| 4.00/4 maps [51:41<00:00] , hybrid_3d2d-seed43__L28s6.tif 18452/18452 patches

[villa] 20251112000002: 4 maps in 3101s -> /vesuvius/output/villa_ensemble_maps/20251112000002
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w087_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w087_ensemble_d+0.png
[done] w087 in 3320s


In [16]:
# --- PHerc1667 w013 (2.4 µm level 2, depth-pooled) ---
render("w013")

20240304141531: 100%|██████████| 4.00/4 maps [33:02<00:00] , hybrid_3d2d-seed43__L28s6.tif 11499/11499 patches

[villa] 20240304141531: 4 maps in 1982s -> /vesuvius/output/villa_ensemble_maps/20240304141531
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w013_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w013_ensemble_d+0.png
[done] w013 in 2098s


In [17]:
# --- PHerc1667 w018 (2.4 µm level 2, depth-pooled; large) ---
render("w018")

20240304144031: 100%|██████████| 4.00/4 maps [3:05:22<00:00] , hybrid_3d2d-seed43__L28s6.tif 57368/57368 patches

[villa] 20240304144031: 4 maps in 11122s -> /vesuvius/output/villa_ensemble_maps/20240304144031
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w018_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/w018_ensemble_d+0.png
[done] w018 in 12687s


In [18]:
# --- PHerc1667 Cr1Fr3 (3.24 µm, resampled) ---
render("cr1fr3")

20231201215900: 100%|██████████| 4.00/4 maps [02:10<00:00] , hybrid_3d2d-seed43__L28s6.tif 756/756 patches

[villa] 20231201215900: 4 maps in 130s -> /vesuvius/output/villa_ensemble_maps/20231201215900
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/cr1fr3_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/cr1fr3_ensemble_d+0.png
[done] cr1fr3 in 138s


In [19]:
# --- PHerc0009B auto_grown 487 (8.64 µm, resampled) ---
render("p9b_487")

20250919125754: 100%|██████████| 4.00/4 maps [18:45<00:00] , hybrid_3d2d-seed43__L28s6.tif 6793/6793 patches

[villa] 20250919125754: 4 maps in 1125s -> /vesuvius/output/villa_ensemble_maps/20250919125754
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p9b_487_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p9b_487_ensemble_d+0.png
[done] p9b_487 in 1198s


In [20]:
# --- PHercParis4 (2.4 µm level 2, depth-pooled) ---
render("paris4")

20231210121321: 100%|██████████| 4.00/4 maps [1:19:02<00:00] , hybrid_3d2d-seed43__L28s6.tif 27727/27727 patches

[villa] 20231210121321: 4 maps in 4742s -> /vesuvius/output/villa_ensemble_maps/20231210121321
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris4_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris4_ensemble_d+0.png
[done] paris4 in 5019s


In [21]:
# --- PHerc0500P2 front ---
render("500P2_front")

20250628074500: 100%|██████████| 4.00/4 maps [09:14<00:00] , hybrid_3d2d-seed43__L28s6.tif 3373/3373 patches

[villa] 20250628074500: 4 maps in 555s -> /vesuvius/output/villa_ensemble_maps/20250628074500
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/500P2_front_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/500P2_front_ensemble_d+0.png
[done] 500P2_front in 596s


In [22]:
# --- PHercParis2 Fr143 (88 keV rescan) ---
render("paris2_fr143")

20230301213755: 100%|██████████| 4.00/4 maps [09:00<00:00] , hybrid_3d2d-seed43__L28s6.tif 3275/3275 patches

[villa] 20230301213755: 4 maps in 540s -> /vesuvius/output/villa_ensemble_maps/20230301213755
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris2_fr143_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris2_fr143_ensemble_d+0.png
[done] paris2_fr143 in 578s


In [23]:
# --- PHercParis2 Fr47 (88 keV rescan) ---
render("paris2_fr47")

20230205142449: 100%|██████████| 4.00/4 maps [02:58<00:00] , hybrid_3d2d-seed43__L28s6.tif 1084/1084 patches

[villa] 20230205142449: 4 maps in 179s -> /vesuvius/output/villa_ensemble_maps/20230205142449
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris2_fr47_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris2_fr47_ensemble_d+0.png
[done] paris2_fr47 in 191s


In [24]:
# --- PHerc51 Cr4 Fr8 (88 keV rescan) ---
render("scroll6_fr8")

20231205222200: 100%|██████████| 4.00/4 maps [03:36<00:00] , hybrid_3d2d-seed43__L28s6.tif 1245/1245 patches

[villa] 20231205222200: 4 maps in 216s -> /vesuvius/output/villa_ensemble_maps/20231205222200
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/scroll6_fr8_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/scroll6_fr8_ensemble_d+0.png
[done] scroll6_fr8 in 230s


In [25]:
# --- PHercParis1 Fr34 (88 keV rescan) ---
render("paris1_fr34")

20230301213423: 100%|██████████| 4.00/4 maps [02:35<00:00] , hybrid_3d2d-seed43__L28s6.tif 921/921 patches

[villa] 20230301213423: 4 maps in 155s -> /vesuvius/output/villa_ensemble_maps/20230301213423
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris1_fr34_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/paris1_fr34_ensemble_d+0.png
[done] paris1_fr34 in 165s


In [26]:
# --- PHerc0343P (8.64 µm, resampled) ---
render("p343")

20250511003658: 100%|██████████| 4.00/4 maps [02:55<00:00] , hybrid_3d2d-seed43__L28s6.tif 1136/1136 patches

[villa] 20250511003658: 4 maps in 175s -> /vesuvius/output/villa_ensemble_maps/20250511003658
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p343_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p343_ensemble_d+0.png
[done] p343 in 187s


In [27]:
# --- PHerc0841 auto_grown (cropped) ---
render("p841")

20260221022814: 100%|██████████| 4.00/4 maps [09:57<00:00] , hybrid_3d2d-seed43__L28s6.tif 3574/3574 patches

[villa] 20260221022814: 4 maps in 597s -> /vesuvius/output/villa_ensemble_maps/20260221022814
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p841_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/training_c40/p841_ensemble_d+0.png
[done] p841 in 636s


In [28]:
# --- unattended: everything not yet rendered ---
render_all()

[skip] w044: already rendered
[skip] w035: already rendered
[skip] w059: already rendered
[skip] w030: already rendered
[skip] w043: already rendered
[skip] w045: already rendered
[skip] w040: already rendered
[skip] w041: already rendered
[skip] w039: already rendered
[skip] w068: already rendered
[skip] w087: already rendered
[skip] w013: already rendered
[skip] w018: already rendered
[skip] cr1fr3: already rendered
[skip] p9b_487: already rendered
[skip] paris4: already rendered
[skip] 500P2_front: already rendered
[skip] seg46527: already rendered
[skip] paris2_fr143: already rendered
[skip] paris2_fr47: already rendered
[skip] scroll6_fr8: already rendered
[skip] paris1_fr34: already rendered
[skip] p343: already rendered
[skip] p841: already rendered
[render_all] 0 failed: []
